Historical study notebook. Its original input is unavailable in this public edition, so this is not a runnable public demo. It may require an input with the original structure and dependencies. This work used supplied learning resources and starter material; retained attribution appears with the notebook content.

# Layout-Aware OCR: PaddleOCR & EasyOCR for Pharmaceutical Documents

**Project 4: Advanced OCR Comparison**

In this notebook, you'll explore two powerful layout-aware OCR engines: **PaddleOCR** and **EasyOCR**. Unlike basic Tesseract, these tools understand document structure and can handle complex pharmaceutical certificates with tables, columns, and form-style layouts.

## What You'll Learn
- How to set up and run PaddleOCR on pharmaceutical SDFs
- How to use EasyOCR for quick, reliable text extraction
- How to visualize results with polygon bounding boxes
- How to compare different OCR engines side-by-side

**Sample Document:** [Pharmaceutical SDF - Page 4 (Certificate of Processing)](https://drive.google.com/file/d/1UvYmChia7K90-xjCsHsGqhixmDrPfhHw/view?usp=sharing)

---

## Part 1: PaddleOCR

PaddleOCR is a powerful, layout-aware OCR engine developed by Baidu. It's especially good at:
- Reading text in tables and complex layouts
- Detecting rotated or skewed text
- Providing precise polygon (not just rectangle) bounding boxes

Let's set it up!

In [ ]:
# One-shot install — no runtime restart needed
!apt-get update
!apt-get install -y tesseract-ocr poppler-utils
!pip install pytesseract paddleocr paddlepaddle easyocr pdf2image pillow opencv-python-headless


In [ ]:
# ============================================
# STEP 2: Import Libraries
# ============================================
#
# WHAT WE'RE DOING:
# Loading all the tools we need for PaddleOCR.
#
# WHY THIS MATTERS:
# Each library has a specific purpose:
# - PaddleOCR: The main OCR engine
# - pdf2image: Converts PDF pages to images
# - PIL: For displaying and drawing on images
# - matplotlib: For showing results nicely
# - cv2/numpy: For image processing
#
# WHAT YOU'LL SEE:
# A success message if everything loaded correctly.
# ============================================

from paddleocr import PaddleOCR
from pdf2image import convert_from_path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import cv2
import numpy as np
from google.colab import files
import os

print("All libraries imported successfully!")

In [ ]:
from google.colab import files
import os
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import cv2
import numpy as np

In [ ]:
# ============================================
# STEP 3: Upload Your Pharmaceutical SDF
# ============================================
#
# WHAT WE'RE DOING:
# Uploading and converting the PDF to an image.
#
# WHY THIS MATTERS:
# Like Tesseract, PaddleOCR works on images.
# We'll convert at 300 DPI for high-quality text recognition.
#
# WHAT YOU'LL SEE:
# A preview of your uploaded pharmaceutical document.
# ============================================

print("Upload your Pharmaceutical SDF (PDF file):")
uploaded = files.upload()

# Get the uploaded filename
pdf_filename = list(uploaded.keys())[0]
pdf_path = f"/content/{pdf_filename}"
print(f"\nUploaded: {pdf_filename}")

# Convert PDF to images (high quality)
print("\nConverting PDF to image...")
images = convert_from_path(pdf_path, dpi=300)
print(f"Converted {len(images)} page(s)")

# Save the first page as an image
image = images[0]
image_path = '/content/pharmaceutical_sdf.png'
image.save(image_path, 'PNG')

# Display the image
plt.figure(figsize=(12, 8))
plt.imshow(image)
plt.title('Uploaded Pharmaceutical SDF')
plt.axis('off')
plt.show()

In [ ]:
# ============================================
# STEP 4: Initialize PaddleOCR
# ============================================
#
# WHAT WE'RE DOING:
# Setting up PaddleOCR with English language support.
#
# WHY THIS MATTERS:
# PaddleOCR needs to download model files the first time.
# The 'use_angle_cls=True' option helps detect rotated text.
#
# WHAT YOU'LL SEE:
# Download progress for model files, then a ready message.
# ============================================

print("Initializing PaddleOCR (this may download model files)...")

# Initialize PaddleOCR
ocr = PaddleOCR(
    use_angle_cls=True,  # Enable text angle detection
    lang='en'            # English language
)

print("\nPaddleOCR initialized and ready!")

# TIP: If you see warnings about GPU, that's okay - it will use CPU instead.

In [ ]:
# ============================================
# STEP 5: Run PaddleOCR on the Pharmaceutical SDF
# ============================================
#
# WHAT WE'RE DOING:
# Running OCR to extract text and bounding boxes.
#
# WHY THIS MATTERS:
# PaddleOCR returns detailed results:
# - The actual text detected
# - Confidence scores (how sure it is)
# - Polygon coordinates (4 corners of the text box)
#
# WHAT YOU'LL SEE:
# Progress messages as it processes the document.
# ============================================

print("Running PaddleOCR on your pharmaceutical document...")
print("(This may take a moment)")

# Run OCR
result = ocr.ocr(image_path)

print("\nOCR complete!")
print(f"Detected {len(result[0]) if result[0] else 0} text regions")

# TIP: result[0] contains all detected text regions from page 1

In [ ]:
# ============================================
# STEP 6: Display Sample Results
# ============================================
#
# WHAT WE'RE DOING:
# Showing you what PaddleOCR found in a readable format.
#
# WHY THIS MATTERS:
# Before drawing boxes, let's see the raw output.
# Each result has: [polygon_box, (text, confidence)]
#
# WHAT YOU'LL SEE:
# A list of detected text with confidence scores.
# ============================================

print("Sample of PaddleOCR Results:")
print("=" * 60)

# Show first 15 results
for i, line in enumerate(result[0][:15]):
    box = line[0]  # Polygon coordinates
    text = line[1][0]  # Detected text
    confidence = line[1][1]  # Confidence score

    print(f"{i+1:2d}. [{confidence:.2%}] {text}")

if len(result[0]) > 15:
    print(f"\n... and {len(result[0]) - 15} more text regions!")

# TIP: Confidence scores above 90% are usually very reliable.

In [ ]:
# ============================================
# STEP 7: Draw Polygon Bounding Boxes
# ============================================
#
# WHAT WE'RE DOING:
# Drawing boxes around every text region PaddleOCR found.
#
# WHY THIS MATTERS:
# Unlike Tesseract's rectangles, PaddleOCR gives us polygons!
# Polygons can follow tilted or skewed text more precisely.
#
# WHAT YOU'LL SEE:
# Your document with red polygon boxes around all detected text.
# ============================================

# Load the original image with OpenCV
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB for display

# Draw polygons for each detected text region
for line in result[0]:
    box = line[0]  # Get the polygon coordinates
    text = line[1][0]  # Get the text
    score = line[1][1]  # Get confidence

    # Only draw for high-confidence detections
    if score > 0.5:
        # Convert box coordinates to integers
        box = [(int(x), int(y)) for x, y in box]

        # Draw the polygon (4-sided shape) in red
        cv2.polylines(img, [np.array(box)], isClosed=True,
                      color=(255, 0, 0), thickness=2)

        # Add the text label near the first corner in blue
        cv2.putText(img, text[:20], box[0],
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

# Display the result
plt.figure(figsize=(15, 10))
plt.imshow(img)
plt.title('PaddleOCR Results - Polygon Bounding Boxes')
plt.axis('off')
plt.show()

# TIP: Notice how polygons can capture tilted text better than rectangles!

---

## Part 2: EasyOCR

EasyOCR is a simpler, faster alternative to PaddleOCR. It's:
- Very easy to install and use
- Good for scanned forms and simple layouts
- Faster to set up than PaddleOCR

Let's try it on the same document!

In [ ]:

# ============================================
# STEP 8: Install and Import EasyOCR
# ============================================
#
# WHAT WE'RE DOING:
# Setting up EasyOCR - it's much simpler than PaddleOCR!
#
# WHY THIS MATTERS:
# Sometimes simpler is better. EasyOCR is:
# - Faster to install
# - Easier to use
# - Still quite accurate
#
# WHAT YOU'LL SEE:
# Installation progress, then a ready message.
# ============================================

!pip install easyocr

import easyocr

print("\nEasyOCR installed and imported!")

In [ ]:
# ============================================
# STEP 9: Initialize EasyOCR Reader
# ============================================
#
# WHAT WE'RE DOING:
# Creating an EasyOCR reader for English text.
#
# WHY THIS MATTERS:
# EasyOCR downloads model files on first use.
# After that, initialization is very fast.
#
# WHAT YOU'LL SEE:
# Model download progress (first time only).
# ============================================

print("Initializing EasyOCR reader...")

# Create a reader for English
reader = easyocr.Reader(['en'])

print("EasyOCR reader ready!")

# TIP: You can add more languages like ['en', 'fr'] for multilingual documents.

In [ ]:
# ============================================
# STEP 10: Run EasyOCR on the Pharmaceutical SDF
# ============================================
#
# WHAT WE'RE DOING:
# Processing the same document with EasyOCR.
#
# WHY THIS MATTERS:
# This lets us compare EasyOCR results with PaddleOCR!
# Each engine may find different things.
#
# WHAT YOU'LL SEE:
# Processing time and number of text regions found.
# ============================================

print("Running EasyOCR on your pharmaceutical document...")

# Run OCR
easy_result = reader.readtext(image_path)

print(f"\nEasyOCR found {len(easy_result)} text regions!")

# TIP: EasyOCR typically runs faster than PaddleOCR.

In [ ]:
# ============================================
# STEP 11: Display EasyOCR Results
# ============================================
#
# WHAT WE'RE DOING:
# Showing what EasyOCR found.
#
# WHY THIS MATTERS:
# EasyOCR returns: (bounding_box, text, confidence)
# The bounding box is 4 corner points.
#
# WHAT YOU'LL SEE:
# A list of detected text with confidence scores.
# ============================================

print("Sample of EasyOCR Results:")
print("=" * 60)

# Show first 15 results
for i, (bbox, text, confidence) in enumerate(easy_result[:15]):
    print(f"{i+1:2d}. [{confidence:.2%}] {text}")

if len(easy_result) > 15:
    print(f"\n... and {len(easy_result) - 15} more text regions!")

In [ ]:
# ============================================
# STEP 12: Visualize EasyOCR Results
# ============================================
#
# WHAT WE'RE DOING:
# Drawing boxes around EasyOCR detections.
#
# WHY THIS MATTERS:
# Visual comparison helps us see differences between engines.
#
# WHAT YOU'LL SEE:
# Your document with red boxes around EasyOCR detections.
# ============================================

# Load a fresh copy of the image
img_easy = image.copy()
draw = ImageDraw.Draw(img_easy)

# Collect high-confidence text
extracted_text = []

for (bbox, text, confidence) in easy_result:
    if confidence > 0.5:  # Only show high-confidence results
        # Get bounding box corners
        top_left = tuple(map(int, bbox[0]))
        bottom_right = tuple(map(int, bbox[2]))

        # Draw rectangle
        draw.rectangle([top_left, bottom_right], outline='red', width=2)

        # Add confidence score
        draw.text((top_left[0], top_left[1] - 20), f"{confidence:.2f}", fill='red')

        extracted_text.append(text)

# Display
plt.figure(figsize=(15, 10))
plt.imshow(img_easy)
plt.title(f'EasyOCR Results - {len(extracted_text)} text segments found')
plt.axis('off')
plt.show()

# TIP: Compare this with the PaddleOCR visualization above!

#Part 3: Tesseract

In [ ]:
import pytesseract
from PIL import Image
tess_text = pytesseract.image_to_string(Image.open(image_path), config=r'--oem 3 -l eng')
print("=== TESSERACT ===\n", tess_text)


---

## Part 3: Side-by-Side Comparison

Now let's compare PaddleOCR,EasyOCR and Tesseract results to see which works better for pharmaceutical documents!

In [ ]:
import pytesseract
from PIL import Image

# Re-run Tesseract in "data" mode to get per-word confidence (image_to_string gives no scores)
tsv = pytesseract.image_to_data(Image.open(image_path),
                                config=r'--oem 3 -l eng',
                                output_type=pytesseract.Output.DICT)
# Keep only real words with a valid confidence
tess_words = [(t, int(c)) for t, c in zip(tsv["text"], tsv["conf"])
              if t.strip() and int(c) >= 0]
tess_count = len(tess_words)
tess_confs = [c/100 for _, c in tess_words]   # Tesseract conf is 0-100, scale to 0-1
tess_avg   = sum(tess_confs)/len(tess_confs) if tess_confs else 0

# Paddle + Easy stats (same as before)
paddle_count = len(result[0]) if result[0] else 0
paddle_confs = [line[1][1] for line in result[0]] if result[0] else []
paddle_avg   = sum(paddle_confs)/len(paddle_confs) if paddle_confs else 0

easy_count = len(easy_result)
easy_confs = [c for (_, _, c) in easy_result]
easy_avg   = sum(easy_confs)/len(easy_confs) if easy_confs else 0

# 3-engine comparison
print("OCR Engine Comparison (3 engines)")
print("=" * 70)
print(f"{'Metric':<25}{'Tesseract':<15}{'PaddleOCR':<15}{'EasyOCR':<15}")
print("-" * 70)
print(f"{'Text regions/words:':<25}{tess_count:<15}{paddle_count:<15}{easy_count:<15}")
print(f"{'Average confidence:':<25}{f'{tess_avg:.2%}':<15}{f'{paddle_avg:.2%}':<15}{f'{easy_avg:.2%}':<15}")
print(f"{'High conf (>90%):':<25}{sum(c>0.9 for c in tess_confs):<15}{sum(c>0.9 for c in paddle_confs):<15}{sum(c>0.9 for c in easy_confs):<15}")
print(f"{'Low conf (<50%):':<25}{sum(c<0.5 for c in tess_confs):<15}{sum(c<0.5 for c in paddle_confs):<15}{sum(c<0.5 for c in easy_confs):<15}")


In [ ]:
key_terms = ["CERTIFICATE","LOT","BATCH","EXPIRATION","DATE",
             "MANUFACTURE","ARTICLE","QUALITY","PRODUCT","VENDOR"]

paddle_texts = [line[1][0].upper() for line in result[0]] if result[0] else []
easy_texts   = [t.upper() for (_, t, _) in easy_result]
tess_upper   = tess_text.upper()   # Tesseract's full string from your earlier cell

print("Key Pharmaceutical Terms Found:")
print("=" * 55)
print(f"{'Term':<15}{'Tesseract':<12}{'PaddleOCR':<12}{'EasyOCR':<12}")
print("-" * 55)
for term in key_terms:
    t = "Found" if term in tess_upper else "--"
    p = "Found" if any(term in x for x in paddle_texts) else "--"
    e = "Found" if any(term in x for x in easy_texts) else "--"
    print(f"{term:<15}{t:<12}{p:<12}{e:<12}")
